# Intro to PandaPower for IDP Project Optimization

In [534]:
try:
    from pandapower.networks.power_system_test_cases import case_ieee30
    from pandapower.run import runpp, runopp
except: 
    %pip install pandapower
    %pip install numba
    from pandapower.networks.power_system_test_cases import case_ieee30
    from pandapower.run import runpp, runopp

### Import IEEE Case 30:
Case 30 is stored in a variable, "net", which consists of several dataframes. 
You can conceptualize a dataframe as being like a spreadsheet: It has named rows and columns, and data is stored in individual cells for each row/column pair. 
By default, the IEEE 30 net has a dataframe for each of the following grid components
- bus (30)
- load (21)
- generator (5)
- shunt (2)
- ext_grid (1)
- line (34)
- transformer (7)
- cost (6)

Additional components, such as for storage, motors, etc. can also be added to the net

In [535]:
net = case_ieee30()

In [536]:
net.gen

,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style
0,None,1,40.0,1.045,NaN,-50.0,40.0,1.0,False,True,0.0,None,True,140.0,0.0,<NA>,False,None
1,None,4,0.0,1.010,NaN,-40.0,40.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
2,None,7,0.0,1.010,NaN,-40.0,10.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
3,None,10,0.0,1.082,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
4,None,12,0.0,1.071,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None


In [537]:
net.load.iloc[0:5,]

,name,bus,p_mw,q_mvar,const_z_p_percent,const_z_q_percent,const_i_p_percent,const_i_q_percent,sn_mva,scaling,in_service,type,controllable
0,None,1,21.7,12.7,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
1,None,2,2.4,1.2,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
2,None,3,7.6,1.6,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
3,None,4,94.2,19.0,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
4,None,6,22.8,10.9,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False


Specific cells are adjustable based on external data

In [538]:
# for example: set the power consumption of load #1 to 5x its starting value
print("Before: " + str(net.load.at[0,'p_mw']))
net.load.at[0,'p_mw'] = 5 * net.load.iloc[0].p_mw
print("After: " + str(net.load.at[0,'p_mw']))

Before: 21.7
After: 108.5


### Power flow analysis
Analysis can be run on net using the runpp function. This adds a new set of dataframes to the net, containing result data for each bus, load, generator, etc.

In [539]:
runpp(net)
net.res_bus[0:5]

,vm_pu,va_degree,p_mw,q_mvar
0,1.060000,0.000000,-354.593394,37.955528
1,1.045000,-7.877331,68.500000,-80.990133
2,1.019979,-8.949676,2.400000,1.200000
3,1.011496,-11.048863,7.600000,1.600000
4,1.010000,-16.387836,94.200000,-16.806864


In [540]:
net.res_gen

,p_mw,q_mvar,va_degree,vm_pu
0,40.0,93.690133,-7.877331,1.045
1,0.0,35.806864,-16.387836,1.010
2,0.0,36.706023,-13.752101,1.010
3,0.0,16.161593,-16.021338,1.082
4,0.0,10.616528,-16.774079,1.071


You can use the result dataframes to check if power flow data is within the defined limits for each object

In [541]:
# For example, buses:
def bus_limits_check():
    err_found = False
    for i in range(len(net.bus)):
        bus = net.bus.iloc[i]
        res = net.res_bus.iloc[i]
        
        if res.vm_pu-bus.min_vm_pu < -0.0001: 
            print(f'Bus #{bus.name}: Result {round(res.vm_pu,4)} V p.u. is less than minimum of {bus.min_vm_pu} V p.u.')
            err_found = True
        elif res.vm_pu-bus.max_vm_pu > 0.0001:
            print(f'Bus #{bus.name}: Result {round(res.vm_pu,4)} V p.u. is greater than maximum of {bus.max_vm_pu} V p.u.')
            err_found = True
        
    if not err_found: print("All buses within limits")

bus_limits_check()

Bus #10: Result 1.082 V p.u. is greater than maximum of 1.06 V p.u.
Bus #12: Result 1.071 V p.u. is greater than maximum of 1.06 V p.u.


For purposes of optimization, grid components such as generators and loads have a boolean property called "controllable". If a component's "controllable" property is true, that means that the component can be changed during the optimization algorithm, to find which values produce the optimal grid conditions. Values where "controllable" is false are those which must be kept fixed.  
By default, all sources are controllable, and all loads are not. 

In [542]:
runopp(net)

gen vm_pu > bus max_vm_pu for gens [3 4]. Setting bus limit for these gens.


Below, you can see that the controllable grids have had their p_mw values changed. Compare the generator results with the generator results from before:

In [543]:
net.res_gen

,p_mw,q_mvar,va_degree,vm_pu
0,38.396460,39.999977,-4.752912,1.038395
1,55.576370,39.999976,-8.847927,1.028954
2,31.100900,9.999984,-6.952433,1.000355
3,21.864394,5.999984,-5.949078,1.045141
4,15.701033,5.999984,-8.609612,1.053313


In [544]:
# proof that bus max/min values are now accounted for
bus_limits_check()

All buses within limits


To demonstrate optimization working if we set one of the generators to a fixed, non-controllable value: 

In [545]:
net.gen.at[0,'p_mw'] = 20
net.gen.at[0, 'controllable'] = False
net.gen

,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style
0,None,1,20.0,1.045,NaN,-50.0,40.0,1.0,False,True,0.0,None,False,140.0,0.0,<NA>,False,None
1,None,4,0.0,1.010,NaN,-40.0,40.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
2,None,7,0.0,1.010,NaN,-40.0,10.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
3,None,10,0.0,1.082,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
4,None,12,0.0,1.071,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None


In [546]:
runpp(net)
bus_limits_check()

Bus #10: Result 1.082 V p.u. is greater than maximum of 1.06 V p.u.
Bus #12: Result 1.071 V p.u. is greater than maximum of 1.06 V p.u.


In [547]:
runopp(net)
bus_limits_check()


gen vm_pu > bus max_vm_pu for gens [3 4]. Setting bus limit for these gens.


All buses within limits


The resulting optimized components have changed based on this new constraint. 

In [548]:
net.res_gen

,p_mw,q_mvar,va_degree,vm_pu
0,20.000000,39.999998,-4.111787,1.045000
1,90.559082,39.999998,-5.787853,1.048978
2,41.453572,9.999997,-5.096205,1.013758
3,25.701383,5.999997,-3.689288,1.056717
4,16.763739,4.189849,-6.867300,1.060000
